<a href="https://colab.research.google.com/github/himanshugithub360/Deep_Learning/blob/main/24_Functional_API/Age_gender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
100% 331M/331M [00:01<00:00, 194MB/s]



In [2]:
import zipfile
zip = zipfile.ZipFile("utkface-new.zip",'r')
zip.extractall()
zip.close()

In [3]:
!pip install tensorflow
import os
import numpy as np
import pandas as pd
import tensorflow as trf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 815.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 49.7 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [4]:
 folder_path = 'utkface_aligned_cropped/UTKFace'

In [5]:
age = []
gender = []
img_path = []

for file in os.listdir(folder_path):
    age.append(int(file.split('_')[0]))
    gender.append(int(file.split('_')[1]))
    img_path.append(file)

In [6]:
len(age)

23708

In [7]:
df = pd.DataFrame({'age':age, 'gender':gender,'img':img_path})

In [8]:
df.sample(5)

,age,gender,img
2944,44,0,44_0_0_20170120134632848.jpg.chip.jpg
5183,1,1,1_1_2_20161219203352244.jpg.chip.jpg
6779,33,1,33_1_3_20170117174243837.jpg.chip.jpg
3210,36,1,36_1_1_20170116023952972.jpg.chip.jpg
492,26,0,26_0_1_20170117170403771.jpg.chip.jpg


In [9]:
train_df = df.sample(frac = 1, random_state=0).iloc[:20000]
test_df = df.sample(frac = 1, random_state=0).iloc[20000:]

In [10]:
train_df.shape

(20000, 3)

In [11]:
test_df.shape

(3708, 3)

In [12]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [13]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [14]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [15]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [16]:
resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [17]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [18]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [19]:
def generator_wrapper(generator):
    while True:
        x, y = next(generator)

        yield x, {
            'age': y[0],
            'gender': y[1]
        }

In [20]:
train_gen = generator_wrapper(train_generator)
test_gen = generator_wrapper(test_generator)

In [ ]:
model.fit(
    train_gen,
    steps_per_epoch=len(train_generator),
    validation_data=test_gen,
    validation_steps=len(test_generator),
    epochs=10
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2018s 3s/step - age_loss: 15.4964 - age_mae: 15.4964 - gender_accuracy: 0.5124 - gender_loss: 0.8968 - loss: 104.2766 - val_age_loss: 14.5689 - val_age_mae: 14.5662 - val_gender_accuracy: 0.5278 - val_gender_loss: 0.6916 - val_loss: 83.0366
Epoch 2/10
131/625 ━━━━━━━━━━━━━━━━━━━━ 23:34 3s/step - age_loss: 15.3856 - age_mae: 15.3856 - gender_accuracy: 0.5206 - gender_loss: 0.6988 - loss: 84.5673